# Day 07 — Model Training & Evaluation

This notebook trains two classification models on the Titanic dataset:

1. **Logistic Regression** — a linear classification baseline.
2. **Random Forest** — an ensemble of decision trees that can learn non-linear patterns.

Both models use the same training/test split and the same preprocessing strategy. The goal is not just to produce an accuracy score, but to compare model behavior using a confusion matrix and a classification report.

## 1. Recreate the Feature Preparation Workflow

A notebook should be reproducible on its own. Therefore, this notebook loads the Titanic dataset and recreates the feature preparation and train/test split from Day 06.

The imputer and scaler are fitted only on the training subset. The test subset remains unseen during preprocessing and model fitting.

In [ ]:
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Load and prepare Titanic features
df = sns.load_dataset("titanic")
df["sex_encoded"] = df["sex"].map({"male": 1, "female": 0})

embarked_encoded = pd.get_dummies(
    df["embarked"],
    prefix="embarked",
    drop_first=True,
    dtype=int
)
embarked_encoded = embarked_encoded.reindex(
    columns=["embarked_Q", "embarked_S"],
    fill_value=0
)
df = pd.concat([df, embarked_encoded], axis=1)
df = df.dropna(subset=["survived", "sex_encoded"])

features = [
    "pclass",
    "age",
    "fare",
    "sex_encoded",
    "embarked_Q",
    "embarked_S"
]
X = df[features].copy()
y = df["survived"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 2. Train the Baseline — Logistic Regression

Logistic Regression is a widely used baseline for binary classification. Despite the word “regression” in its name, it predicts the probability of belonging to a class.

Here, the target is `survived`:

- `0`: passenger did not survive.
- `1`: passenger survived.

The model learns relationships between the selected features and the probability of survival. The `random_state` helps keep the solver's behavior reproducible where applicable.

In [ ]:
log_model = LogisticRegression(random_state=42, max_iter=1000)
log_model.fit(X_train_processed, y_train)

y_pred_log = log_model.predict(X_test_processed)

print("=== Logistic Regression Results ===")
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_log))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_log, zero_division=0))

## 3. Train the Random Forest Classifier

Random Forest is an ensemble learning method. It trains multiple decision trees on varied samples and feature subsets, then combines their predictions.

The main parameters in this baseline are:

- `n_estimators=100`: build 100 decision trees.
- `max_depth=5`: limit each tree's depth to reduce uncontrolled complexity.
- `random_state=42`: make the experiment reproducible.

Unlike Logistic Regression, Random Forest does not depend on standardized feature scales. We keep the same preprocessed input here so that the experiment follows one consistent workflow; in a later comparison, we can test a tree model without scaling as well.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)
rf_model.fit(X_train_processed, y_train)

y_pred_rf = rf_model.predict(X_test_processed)

print("=== Random Forest Results ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, zero_division=0))

## 4. Compare the Models

Accuracy gives the proportion of correct predictions, but it does not tell the whole story. The classification report adds:

- **Precision:** among records predicted as a class, how many were correct?
- **Recall:** among actual records in a class, how many did the model find?
- **F1-score:** a harmonic balance between precision and recall.
- **Support:** the number of actual test examples in each class.

The confusion matrix shows the counts of correct and incorrect predictions for each class. For a survival problem, it is important to inspect both classes rather than relying on accuracy alone.

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_log),
        accuracy_score(y_test, y_pred_rf)
    ]
})

display(comparison.sort_values("Accuracy", ascending=False).reset_index(drop=True))

## Engineering Takeaways

- Establish a simple baseline before experimenting with more complex models.
- Evaluate on held-out test data that was not used to fit the model or preprocessing steps.
- Accuracy is useful, but precision, recall, F1-score, and the confusion matrix provide a more complete picture.
- Logistic Regression is a useful interpretable baseline; Random Forest can capture more complex feature interactions.
- A higher accuracy score does not automatically mean a model is better for every use case. The cost of false positives and false negatives should influence model selection.
- Model performance is dataset- and split-dependent. The printed metrics from the executed notebook are the evidence; avoid claiming a result before running the experiment.

> **Next engineering step:** Use cross-validation and compare additional metrics such as ROC-AUC. Then tune hyperparameters on training/validation data while keeping the final test set untouched.